# 6.35 - CertCF Shrinkage Ablation on HELOC

Minimal ablation for adaptive epsilon shrinkage: same HELOC support/query split, CertCF without shrinkage vs with shrinkage, and epsilon-certification violin plots.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from certcf import NearestOppositeClassClearanceStrategy
from counterfactuals.datasets.loaders import HELOCDataset
from counterfactuals.methods.certcf import CertCF
from scripts.benchmark import _build_torch_model_from_checkpoint

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)
plt.rcParams.update({'figure.dpi': 120})

DATASET = 'heloc'
SEED = 42
N_SUPPORT = 500
N_QUERIES = 50
ALPHAS = [0.01] + [round(float(a), 2) for a in np.arange(0.05, 1.00, 0.05)] + [0.99]
CLASSIFICATION_MARGIN = 1.0e-4
VIOLIN_BW_METHOD = 0.25  # smaller = sharper violins; larger = smoother violins
NORM = 1
DEVICE = 'auto'

CHECKPOINT = ROOT / 'checkpoints/heloc_classifier/best.ckpt'
rng = np.random.default_rng(SEED)

print({'dataset': DATASET, 'n_support': N_SUPPORT, 'n_queries': N_QUERIES, 'n_alphas': len(ALPHAS)})

## Load Data

In [ ]:
def stratified_indices(labels: np.ndarray, n_total: int, rng: np.random.Generator) -> np.ndarray:
    labels = np.asarray(labels, dtype=np.int64)
    classes = np.unique(labels)
    base = int(n_total) // len(classes)
    remainder = int(n_total) - base * len(classes)
    chosen = []
    for pos, cls in enumerate(classes):
        idx = np.flatnonzero(labels == cls)
        take = min(len(idx), base + int(pos < remainder))
        chosen.append(rng.choice(idx, size=take, replace=False))
    out = np.concatenate(chosen)
    rng.shuffle(out)
    return out

dataset = HELOCDataset(data_dir=str(ROOT / 'data'), seed=SEED)
dataset.load()
x_train_full, y_train = dataset.get_train()
x_test, y_test = dataset.get_test()
spec = dataset.spec

model = _build_torch_model_from_checkpoint(
    checkpoint=str(CHECKPOINT),
    device=DEVICE,
    dataset_module=DATASET,
    hidden_dims=[64, 32],
    dropout=0.2,
)

y_train_pred = model.predict(x_train_full).astype(np.int64)
support_idx = stratified_indices(y_train_pred, N_SUPPORT, rng)
x_support = x_train_full[support_idx]
y_support = y_train_pred[support_idx]

y_test_pred = model.predict(x_test).astype(np.int64)
query_idx = rng.choice(np.arange(len(x_test)), size=min(N_QUERIES, len(x_test)), replace=False)
x_query = x_test[query_idx]
target_query = 1 - y_test_pred[query_idx]

display(pd.DataFrame([
    {
        'dataset': DATASET,
        'encoded_dim': x_train_full.shape[1],
        'train_rows': len(x_train_full),
        'test_rows': len(x_test),
        'support_rows': len(x_support),
        'query_rows': len(x_query),
        'train_label_agreement': np.mean(y_train_pred == y_train),
    }
]).round(4))
print('support counts by model prediction:', dict(zip(*np.unique(y_support, return_counts=True))))

## Build CertCF Variants

In [ ]:
def center_slack_from_bounds(A: np.ndarray, b: np.ndarray, center: np.ndarray) -> float:
    A = np.asarray(A, dtype=np.float64).reshape(-1, len(center))
    b = np.asarray(b, dtype=np.float64).reshape(-1)
    if len(b) == 0:
        return float('inf')
    return float(np.min(A @ np.asarray(center, dtype=np.float64) + b - CLASSIFICATION_MARGIN))


def atlas_polytope_diagnostics(method: CertCF, variant: str, alpha: float) -> pd.DataFrame:
    rows = []
    atlas = method.atlas
    for label in atlas.class_labels:
        bd = atlas.bounds[int(label)]
        eps = np.asarray(bd['eps'], dtype=float)
        eps_initial = np.asarray(bd.get('eps_initial', eps), dtype=float)
        shrinks = np.asarray(bd.get('adaptive_eps_n_shrinks', np.zeros_like(eps)), dtype=float)
        stored_slack = bd.get('adaptive_eps_center_slack')
        stored_certified = bd.get('adaptive_eps_center_certified')
        for idx, center in enumerate(np.asarray(bd['X'], dtype=np.float64)):
            slack = (
                float(stored_slack[idx])
                if stored_slack is not None
                else center_slack_from_bounds(bd['lA'][idx], bd['lbias'][idx], center)
            )
            certified = (
                bool(stored_certified[idx])
                if stored_certified is not None
                else bool(slack >= -1.0e-6)
            )
            rows.append({
                'variant': variant,
                'alpha': float(alpha),
                'class_label': int(label),
                'polytope_idx': int(idx),
                'eps_initial': float(eps_initial[idx]),
                'eps_final': float(eps[idx]),
                'center_certified': certified,
                'status': 'certified' if certified else 'not certified',
                'n_shrinks': float(shrinks[idx]),
                'center_slack': slack,
            })
    return pd.DataFrame(rows)


def query_summary(method: CertCF, variant: str, alpha: float) -> dict:
    t0 = time.perf_counter()
    results = method.generate_batch(x_query, target_class=target_query)
    total_s = time.perf_counter() - t0
    l1 = [
        float(np.linalg.norm(result.x_cf - x_query[i], ord=1))
        for i, result in enumerate(results)
        if result.x_cf is not None and result.success
    ]
    return {
        'variant': variant,
        'alpha': float(alpha),
        'validity_pct': 100.0 * float(np.mean([r.success for r in results])),
        'l1_mean': float(np.mean(l1)) if l1 else np.nan,
        'query_time_s': total_s / max(1, len(results)),
    }


def build_method(alpha: float, adaptive_eps: bool) -> CertCF:
    return CertCF(
        model=model,
        norm=NORM,
        distance_norm=NORM,
        lirpa_method='backward',
        eps_strategy=NearestOppositeClassClearanceStrategy(alpha=float(alpha)),
        batch_size=128,
        ohe_slices=list(spec.categorical_slices),
        default_query_method='nearest_anchor',
        query_k_candidates=5,
        k_per_class=None,
        classification_margin=CLASSIFICATION_MARGIN,
        random_seed=SEED,
        adaptive_eps=adaptive_eps,
        adaptive_eps_shrink_factor=0.5,
        adaptive_eps_max_shrinks=8,
        adaptive_eps_min=1.0e-6,
        adaptive_eps_center_tol=1.0e-6,
        adaptive_eps_binary_search_steps=0,
    )


variants = [('no shrinkage', False), ('with shrinkage', True)]
polytope_frames = []
query_rows = []

for variant, adaptive in variants:
    for alpha in ALPHAS:
        print(f'building {variant}, alpha={alpha}')
        method = build_method(alpha, adaptive_eps=adaptive)
        method.fit(x_train=x_support, y_train=y_support)
        polytope_frames.append(atlas_polytope_diagnostics(method, variant, alpha))
        query_rows.append(query_summary(method, variant, alpha))

POLYTOPE_DF = pd.concat(polytope_frames, ignore_index=True)
QUERY_SUMMARY_DF = pd.DataFrame(query_rows)
display(QUERY_SUMMARY_DF.round(4))

## Epsilon Certification Plots

In [ ]:
def plot_epsilon_violin(polytope_df: pd.DataFrame, variant: str):
    plot_df = polytope_df[polytope_df['variant'].eq(variant)].copy()
    status_order = ['certified', 'not certified']
    colors = {'certified': '#087F5B', 'not certified': '#C92A2A'}
    offsets = {'certified': -0.18, 'not certified': 0.18}

    fig, ax = plt.subplots(figsize=(12.5, 5.2))
    handles = []
    for status in status_order:
        data, positions = [], []
        for alpha_idx, alpha in enumerate(ALPHAS):
            values = plot_df.loc[
                np.isclose(plot_df['alpha'].astype(float), float(alpha)) & plot_df['status'].eq(status),
                'eps_final',
            ].to_numpy(dtype=float)
            values = values[np.isfinite(values) & (values > 0.0)]
            if len(values) == 0:
                continue
            data.append(values)
            positions.append(alpha_idx + offsets[status])
        if not data:
            continue

        violins = ax.violinplot(
            data,
            positions=positions,
            widths=0.30,
            showmeans=False,
            showmedians=True,
            showextrema=False,
            bw_method=VIOLIN_BW_METHOD,
        )
        for body in violins['bodies']:
            body.set_facecolor(colors[status])
            body.set_edgecolor(colors[status])
            body.set_alpha(0.35)
            body.set_linewidth(0.8)
        violins['cmedians'].set_color(colors[status])
        violins['cmedians'].set_linewidth(1.5)
        handles.append(plt.Line2D([0], [0], color=colors[status], linewidth=6, alpha=0.35, label=status))

    ax.set_title(f'HELOC: epsilon radius distribution ({variant})')
    ax.set_xlabel('alpha')
    ax.set_ylabel('epsilon (log scale)')
    ax.set_yscale('log')
    ax.set_xticks(range(len(ALPHAS)))
    ax.set_xticklabels([str(a) for a in ALPHAS], rotation=45, ha='right')
    ax.grid(True, axis='y', alpha=0.25, which='major')
    ax.grid(True, axis='y', alpha=0.12, which='minor')
    if handles:
        ax.legend(handles=handles, frameon=False, loc='upper left')
    fig.tight_layout()
    return fig, ax


plot_epsilon_violin(POLYTOPE_DF, 'no shrinkage');
plot_epsilon_violin(POLYTOPE_DF, 'with shrinkage');

## Overlay Shrinkage Effect


In [ ]:
def plot_overlay_final_epsilon(polytope_df: pd.DataFrame):
    plot_df = polytope_df.copy()
    groups = [
        ('no shrinkage', 'certified', -0.30, '#087F5B', 'no shrinkage | certified'),
        ('no shrinkage', 'not certified', -0.10, '#D62728', 'no shrinkage | not certified'),
        ('with shrinkage', 'certified', 0.10, '#1F77B4', 'with shrinkage | certified'),
        ('with shrinkage', 'not certified', 0.30, '#E67700', 'with shrinkage | not certified'),
    ]

    fig, ax = plt.subplots(figsize=(12.8, 5.4))
    handles = []
    for variant, status, offset, color, label in groups:
        data, positions, medians = [], [], []
        for alpha_idx, alpha in enumerate(ALPHAS):
            values = plot_df.loc[
                plot_df['variant'].eq(variant)
                & plot_df['status'].eq(status)
                & np.isclose(plot_df['alpha'].astype(float), float(alpha)),
                'eps_final',
            ].to_numpy(dtype=float)
            values = values[np.isfinite(values) & (values > 0.0)]
            if len(values) == 0:
                continue
            data.append(values)
            positions.append(alpha_idx + offset)
            medians.append(float(np.median(values)))
        if not data:
            continue

        violins = ax.violinplot(
            data,
            positions=positions,
            widths=0.18,
            showmeans=False,
            showmedians=True,
            showextrema=False,
            bw_method=VIOLIN_BW_METHOD,
        )
        for body in violins['bodies']:
            body.set_facecolor(color)
            body.set_edgecolor(color)
            body.set_alpha(0.30)
            body.set_linewidth(0.8)
        violins['cmedians'].set_color(color)
        violins['cmedians'].set_linewidth(1.3)
        ax.plot(
            positions,
            medians,
            color=color,
            linewidth=1.5,
            marker='o',
            markersize=2.8,
            alpha=0.90,
            zorder=4,
        )
        handles.append(plt.Line2D([0], [0], color=color, linewidth=1.8, marker='o', markersize=3, alpha=0.90, label=label))

    ax.set_xlabel('alpha')
    ax.set_ylabel('final epsilon (log scale)')
    ax.set_yscale('log')
    ax.set_xticks(range(len(ALPHAS)))
    ax.set_xticklabels([str(a) for a in ALPHAS], rotation=45, ha='right')
    ax.grid(True, axis='y', alpha=0.25, which='major')
    ax.grid(True, axis='y', alpha=0.12, which='minor')
    ax.legend(handles=handles, frameon=False, loc='upper left', ncol=2, fontsize=13)
    fig.tight_layout()
    return fig, ax


plot_overlay_final_epsilon(POLYTOPE_DF);


## Export Violin Geometry for LaTeX

This exports precomputed violin polygons and median segments so PGFPlots only has to draw vector shapes.

In [ ]:
def simple_gaussian_kde_1d(values: np.ndarray, grid: np.ndarray, bw_factor: float = VIOLIN_BW_METHOD) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.zeros_like(grid, dtype=float)
    if len(values) == 1 or np.nanstd(values) == 0:
        center = float(values[0])
        scale = max(abs(center) * 0.05, 1e-3)
        return np.exp(-0.5 * ((grid - center) / scale) ** 2) / (scale * np.sqrt(2.0 * np.pi))

    std = float(np.std(values, ddof=1))
    scott = len(values) ** (-1.0 / 5.0)
    bandwidth = max(float(bw_factor) * scott * std, 1e-12)
    z = (grid[:, None] - values[None, :]) / bandwidth
    density = np.exp(-0.5 * z ** 2).mean(axis=1) / (bandwidth * np.sqrt(2.0 * np.pi))
    return density


def build_latex_violin_data(
    polytope_df: pd.DataFrame,
    variant: str,
    variant_key: str,
    variant_id: int,
    max_width: float = 0.32,
    n_grid: int = 160,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    plot_df = polytope_df[polytope_df['variant'].eq(variant)].copy()
    status_offsets = {'certified': -0.18, 'not certified': 0.18}
    status_ids = {'certified': 0, 'not certified': 1}
    polygon_rows = []
    median_rows = []

    for alpha_idx, alpha in enumerate(ALPHAS):
        for status, offset in status_offsets.items():
            values = plot_df.loc[
                np.isclose(plot_df['alpha'].astype(float), float(alpha)) & plot_df['status'].eq(status),
                'eps_final',
            ].to_numpy(dtype=float)
            values = values[np.isfinite(values) & (values > 0.0)]
            if len(values) == 0:
                continue

            if len(values) == 1:
                y_grid = np.array([values[0] * 0.97, values[0], values[0] * 1.03], dtype=float)
            else:
                y_grid = np.linspace(values.min(), values.max(), int(n_grid))
            density = simple_gaussian_kde_1d(values, y_grid)
            if np.nanmax(density) <= 0:
                continue
            half_width = max_width * density / np.nanmax(density)
            center_x = float(alpha_idx) + float(offset)
            left_x = center_x - half_width
            right_x = center_x + half_width

            status_id = int(status_ids[status])
            plot_id = int(variant_id) * 100 + int(alpha_idx) * 2 + status_id
            polygon_id = f'{variant_key}_alpha_{alpha:g}_{status.replace(" ", "_")}'
            xs = np.r_[left_x, right_x[::-1], left_x[0]]
            ys = np.r_[y_grid, y_grid[::-1], y_grid[0]]
            for order, (x, y) in enumerate(zip(xs, ys)):
                polygon_rows.append({
                    'variant': variant_key,
                    'polygon_id': polygon_id,
                    'plot_id': int(plot_id),
                    'variant_id': int(variant_id),
                    'status_id': int(status_id),
                    'alpha': float(alpha),
                    'alpha_idx': int(alpha_idx),
                    'status': status,
                    'order': int(order),
                    'x': float(x),
                    'epsilon': float(y),
                })

            median = float(np.median(values))
            median_rows.extend([
                {
                    'variant': variant_key,
                    'segment_id': polygon_id,
                    'plot_id': int(plot_id),
                    'variant_id': int(variant_id),
                    'status_id': int(status_id),
                    'alpha': float(alpha),
                    'alpha_idx': int(alpha_idx),
                    'status': status,
                    'order': 0,
                    'x': float(center_x - max_width * 0.22),
                    'epsilon': median,
                },
                {
                    'variant': variant_key,
                    'segment_id': polygon_id,
                    'plot_id': int(plot_id),
                    'variant_id': int(variant_id),
                    'status_id': int(status_id),
                    'alpha': float(alpha),
                    'alpha_idx': int(alpha_idx),
                    'status': status,
                    'order': 1,
                    'x': float(center_x + max_width * 0.22),
                    'epsilon': median,
                },
            ])

    return pd.DataFrame(polygon_rows), pd.DataFrame(median_rows)


PAPER_ROOT = Path('/home/gabrielepintus/Documents/github/CertCF')
PLOT_DATA_DIR = PAPER_ROOT / 'sections/plots/data'
PLOT_DATA_DIR.mkdir(parents=True, exist_ok=True)

polygon_frames = []
median_frames = []
for variant, variant_key, variant_id in [('no shrinkage', 'no_shrinkage', 0), ('with shrinkage', 'with_shrinkage', 1)]:
    polygons, medians = build_latex_violin_data(POLYTOPE_DF, variant=variant, variant_key=variant_key, variant_id=variant_id)
    polygon_frames.append(polygons)
    median_frames.append(medians)

LATEX_VIOLIN_POLYGONS_DF = pd.concat(polygon_frames, ignore_index=True)
LATEX_VIOLIN_MEDIANS_DF = pd.concat(median_frames, ignore_index=True)



def _tex_float(x: float) -> str:
    return f'{float(x):.8g}'


def write_pgfplots_coordinate_fragments(
    polygons_df: pd.DataFrame,
    medians_df: pd.DataFrame,
    out_dir: Path,
) -> None:
    """Write compact \addplot coordinate fragments to avoid PGFPlots CSV filtering."""
    out_dir.mkdir(parents=True, exist_ok=True)
    style_by_status = {
        'certified': 'draw=shrinkCertified, fill=shrinkCertified, fill opacity=0.30, line width=0.2pt',
        'not certified': 'draw=shrinkFailed, fill=shrinkFailed, fill opacity=0.24, line width=0.2pt',
    }
    median_style_by_status = {
        'certified': 'shrinkCertified, line width=0.55pt',
        'not certified': 'shrinkFailed, line width=0.55pt',
    }

    for variant_key in ['no_shrinkage', 'with_shrinkage']:
        lines = []
        variant_polygons = polygons_df[polygons_df['variant'].eq(variant_key)].copy()
        variant_medians = medians_df[medians_df['variant'].eq(variant_key)].copy()

        for _, meta in variant_polygons[['plot_id', 'status']].drop_duplicates().sort_values('plot_id').iterrows():
            plot_id = int(meta['plot_id'])
            status = str(meta['status'])
            poly = variant_polygons[variant_polygons['plot_id'].eq(plot_id)].sort_values('order')
            med = variant_medians[variant_medians['plot_id'].eq(plot_id)].sort_values('order')

            coords = ' '.join(f'({_tex_float(x)},{_tex_float(y)})' for x, y in zip(poly['x'], poly['epsilon']))
            lines.append(f'\\addplot[{style_by_status[status]}] coordinates {{{coords}}};')

            if not med.empty:
                med_coords = ' '.join(f'({_tex_float(x)},{_tex_float(y)})' for x, y in zip(med['x'], med['epsilon']))
                lines.append(f'\\addplot[{median_style_by_status[status]}] coordinates {{{med_coords}}};')

        (out_dir / f'heloc_epsilon_violin_{variant_key}_coords.tex').write_text('\n'.join(lines) + '\n')

LATEX_VIOLIN_POLYGONS_DF.to_csv(PLOT_DATA_DIR / 'heloc_epsilon_violin_polygons.csv', index=False)
LATEX_VIOLIN_MEDIANS_DF.to_csv(PLOT_DATA_DIR / 'heloc_epsilon_violin_medians.csv', index=False)
write_pgfplots_coordinate_fragments(LATEX_VIOLIN_POLYGONS_DF, LATEX_VIOLIN_MEDIANS_DF, PLOT_DATA_DIR)

print(PLOT_DATA_DIR / 'heloc_epsilon_violin_polygons.csv')
print(PLOT_DATA_DIR / 'heloc_epsilon_violin_medians.csv')
display(LATEX_VIOLIN_POLYGONS_DF.head())


## Export PDF Plots


In [ ]:
PAPER_ROOT = Path('/home/gabrielepintus/Documents/github/CertCF')
PLOT_DIR = PAPER_ROOT / 'sections/plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

fig, _ = plot_epsilon_violin(POLYTOPE_DF, 'no shrinkage')
fig.savefig(PLOT_DIR / 'heloc_epsilon_no_shrinkage.pdf', bbox_inches='tight')
plt.close(fig)

fig, _ = plot_epsilon_violin(POLYTOPE_DF, 'with shrinkage')
fig.savefig(PLOT_DIR / 'heloc_epsilon_with_shrinkage.pdf', bbox_inches='tight')
plt.close(fig)

fig, _ = plot_overlay_final_epsilon(POLYTOPE_DF)
fig.savefig(PLOT_DIR / 'heloc_epsilon_overlay_shrinkage.pdf', bbox_inches='tight')
plt.close(fig)

print(PLOT_DIR / 'heloc_epsilon_no_shrinkage.pdf')
print(PLOT_DIR / 'heloc_epsilon_with_shrinkage.pdf')
print(PLOT_DIR / 'heloc_epsilon_overlay_shrinkage.pdf')
